# Praktikum Kecerdasan Artifisial Lanjut


---

## Bab 5. Klasifikasi Decision Tree dan XGBoost

Nama: Erza Hanif Pramudita Hanggara

NIM: 245150200111038


### Decision Tree

#### 1) Import Data

Praktikum kali ini menggunakan dataset [Car Evaluation Dataset](https://archive.ics.uci.edu/ml/datasets/Car+Evaluation) dari UCI Machine Learning Repository. Dataset ini telah digunakan pada praktikum sebelumnya. Detail fitur dapat Anda pelajari pada link yang tersedia.

Unduh dataset yang akan digunakan pada praktikum kali ini. Anda dapat menggunakan aplikasi wget untuk mendowload dataset dan menyimpannya dalam Google Colab. Jalankan cell di bawah ini untuk mengunduh dataset

In [ ]:
! wget https://archive.ics.uci.edu/static/public/19/car+evaluation.zip

--2026-03-27 07:00:45--  https://archive.ics.uci.edu/static/public/19/car+evaluation.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘car+evaluation.zip’

car+evaluation.zip      [ <=>                ]   6.19K  --.-KB/s    in 0s      

2026-03-27 07:00:46 (87.8 MB/s) - ‘car+evaluation.zip’ saved [6342]



In [ ]:
import zipfile
import pandas as pd

# Ekstrak file zip
with zipfile.ZipFile('car+evaluation.zip', 'r') as zip_ref:
  zip_ref.extractall('car_evaluation')

Setelah dataset berhasil diunduh, langkah berikutnya adalah membaca dataset dengan memanfaatkan fungsi **readcsv** dari library pandas. Lakukan pembacaan berkas csv ke dalam dataframe dengan nama **data** menggunakan fungsi **readcsv**. Jangan lupa untuk melakukan import library pandas terlebih dahulu


In [ ]:
# Membaca dataset (tidak ada header, jadi kita tambahkan manual)
col_names = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']
data = pd.read_csv('car_evaluation/car.data', names=col_names)



Cek isi dataset Anda dengan menggunakan perintah **head()**

In [ ]:
data.head()

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


#### 2) Membagi data menjadi data latih dan data uji

Metode pembelajaran mesin memerlukan dua jenis data :


1.   Data latih : Digunakan untuk proses training metode klasifikasi
2.   Data uji : Digunakan untuk proses evaluasi metode klasifikasi

Data uji dan data latih perlu dibuat terpisah (mutualy exclusive) agar hasil evaluasi lebih akurat.

Data uji dan data latih dapat dibuat dengan cara membagi dataset dengan rasio tertentu, misalnya 80% data latih dan 20% data uji.

Library Scikit-learn memiliki fungsi [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) pada modul **model_selection** untuk membagi dataset menjadi data latih dan data uji. Bagilah dataset anda menjadi dua, yaitu **data_latih** dan **data_uji**. Agar pengacakan data dilakukan secara konstan, parameter **random_state** diisi dengan nilai integer tertentu, pada praktikum ini diset 101. Kemudian, nilai indeks pada data latih dan data uji diatur ulang agar berurutan nilainya


In [ ]:
from sklearn.model_selection import train_test_split
data_latih, data_uji = train_test_split(data, test_size=0.2, random_state=101)
data_latih = data_latih.reset_index(drop=True)
data_uji = data_uji.reset_index(drop=True)

Tampilkan banyaknya data pada **data_latih** dan **data_uji**. Seharusnya **data_latih** terdiri dari 208 data, dan **data_uji** terdiri dari 52 data

In [ ]:
print(data_latih.shape[0])
print(data_uji.shape[0])

1382
346


#### 3) Menghitung Gini

Nilai Gini merupakan salah satu kriteria penentu variabel apa yang akan digunakan untuk membentuk cabang pada decision tree. Variabel dengan nilai Gini terbesar akan digunakan sebagai pembentukan cabang

Buatlah fungsi bernama **hitung_gini** yang berfungsi menghitung nilai Gini dari suatu nilai pada sebuah variabel

In [ ]:
import numpy as np

In [ ]:
def hitung_gini(kolom_kelas):
  elemen,banyak = np.unique(kolom_kelas,return_counts=True)
  nilai_gini = 1 - np.sum([(banyak[i]/np.sum(banyak))**2 for i in range(len(elemen))])
  return nilai_gini

Buatlah fungsi bernama **gini_split** yang digunakan untuk menghitung nilai Gini keseluruhan dari sebuah variabel.

In [ ]:
def gini_split(data, nama_fitur_split, nama_fitur_kelas):
  nilai,banyak = np.unique(data[nama_fitur_split],return_counts=True)
  gini_split = np.sum([(banyak[i]/np.sum(banyak))*hitung_gini(data.where(
      data[nama_fitur_split]==nilai[i]).dropna()[nama_fitur_kelas]) for i
      in range(len(nilai))])
  return gini_split

Ujilah fungsi **gini_split** menggunakan data_latih pada variabel **buying** dan variabel kelas bernama **class**.

In [ ]:
gini_split(data_latih, "buying", "class")

np.float64(0.4498424838345615)

#### 4) Pembentukan pohon

Pembentukan pohon dilakukan secara rekursif. Seperti metode rekursif pada umumnya, perlu ditentukan kondisi berhenti terlebih dahulu. Kondisi berhenti pada pembentukan pohon adalah:


1.   Jika hanya ada satu kelas pada data, kembalikan kelas tersebut
2.   Jika fitur data  = 0 (tidak ada fitur yang tersisa), kembalikan kelas dari parent
3. Jika data kosong (tidak ada data), kembalikan kelas dengan frekuensi terbanyak

Selain kondisi berhenti tersebut, dilakukan pembentukan pohon secara rekursif menggunakan fungsi **buat_tree**.



In [ ]:
def buat_tree(data,data_awal, daftar_fitur, nama_fitur_kelas,kelas_parent_node=None):
  #jika hanya ada satu kelas pada data
  if len(np.unique(data[nama_fitur_kelas])) <= 1:
    return np.unique(data[nama_fitur_kelas])[0]
  #jika data kosong
  elif len(data) == 0:
    return np.unique(data_awal[nama_fitur_kelas])[
      np.argmax(np.unique(data_awal[nama_fitur_kelas],return_counts=True)[1])]

  #jika tidak ada fitur yang terisa
  elif len(daftar_fitur) == 0:
    return kelas_parent_node
  else:
    kelas_parent_node = np.unique(data[nama_fitur_kelas])[
      np.argmax(np.unique(data[nama_fitur_kelas],return_counts=True)[1])]
    nilai_split = [gini_split(data,fitur,nama_fitur_kelas) for fitur in daftar_fitur]
    index_fitur_terbaik = np.argmin(nilai_split)
    fitur_terbaik = daftar_fitur[index_fitur_terbaik]
    tree = {fitur_terbaik:{}}
    daftar_fitur = [i for i in daftar_fitur if i != fitur_terbaik]
    for nilai in np.unique(data[fitur_terbaik]):
      sub_data = data.where(data[fitur_terbaik] == nilai).dropna()
      subtree = buat_tree(sub_data,data_awal,daftar_fitur,nama_fitur_kelas,
                          kelas_parent_node)
      tree[fitur_terbaik][nilai]=subtree
  return(tree)

Buatlah tree menggunakan data latih yang tersedia

In [ ]:
tree = buat_tree(data_latih,data_latih,data_latih.columns[:-1],'class')

Tampilkan tree yang terbentuk. Gunakan library **pprint** untuk menampilkan dictionary secara teratur.

In [ ]:
from pprint import pprint
pprint(tree)

{'safety': {'high': {'persons': {'2': 'unacc',
                                 '4': {'buying': {'high': {'maint': {'high': 'acc',
                                                                     'low': 'acc',
                                                                     'med': 'acc',
                                                                     'vhigh': 'unacc'}},
                                                  'low': {'maint': {'high': {'lug_boot': {'big': 'vgood',
                                                                                          'med': {'doors': {'2': 'acc',
                                                                                                            '3': 'acc',
                                                                                                            '4': 'vgood'}},
                                                                                          'small': 'acc'}},
                                    

#### 5) Proses Prediksi

Proses prediksi kelas pada data uji dilakukan dengan melakukan *tree traversal* sampai menemui leaf.

In [ ]:
def prediksi(data_uji,tree):
  for key in list(data_uji.keys()):
    if key in list(tree.keys()):
      try:
        hasil = tree[key][data_uji[key]]
      except:
        return 1
      hasil = tree[key][data_uji[key]]
      if isinstance(hasil,dict):
        return prediksi(data_uji,hasil)
      else:
        return hasil

#### 6) Proses Pengujian
Lakukan pengujian menggunakan data uji. Kelas pada data uji perlu dihapus dan data uji perlu diubah menjadi dictionary

In [ ]:
data_uji_dict = data_uji.iloc[:,:-1].to_dict(orient='records')

Lakukan pengujian terhadap keseluruhan data uji menggunakan looping.

In [ ]:
hasil_prediksi_total = []
for i in range(len(data_uji_dict)):
  hasil_prediksi = prediksi(data_uji_dict[i],tree)
  hasil_prediksi_total.append(hasil_prediksi)

Bandingkan hasil prediksi dengan label sebenarnya. Hitunglah banyaknya data uji yang memiliki kelas prediksi sama dengan kelas sebenarnya

In [ ]:
print("Total prediksi benar: ",sum(hasil_prediksi_total==data_uji['class']))
print("Total prediksi salah: ",sum(hasil_prediksi_total!=data_uji['class']))
print("Total data: ", len(data_uji))

Total prediksi benar:  298
Total prediksi salah:  48
Total data:  346


## XGBoost

#### 1) Instalasi module/package/library XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

#### 2) Impor Data

In [ ]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df = pd.read_csv('car_evaluation/car.data', names=col_names)
df_new = df.apply(label_encoder.fit_transform)

X_train, X_test, y_train, y_test = train_test_split(df_new.loc[:,'buying':'safety'], df_new['class'], test_size=.2)
#create model instance
bst = XGBClassifier(n_estimators=2, max_depth=2, learning_rate=1, objective='binary:logistic')

#### 3) Pembuatan Classifier dan Model

In [ ]:
bst.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=2,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=2,
              n_jobs=None, num_parallel_tree=None, ...)

#### 4) Proses Pengujian/Prediksi

In [ ]:
y_pred = bst.predict(X_test)

####5) Tampilkan Hasil

In [ ]:
y_label = label_encoder.inverse_transform(y_pred)
print(y_label)

['unacc' 'unacc' 'unacc' 'acc' 'unacc' 'unacc' 'unacc' 'acc' 'unacc' 'acc'
 'acc' 'acc' 'unacc' 'acc' 'unacc' 'acc' 'acc' 'unacc' 'unacc' 'unacc'
 'acc' 'unacc' 'acc' 'unacc' 'acc' 'acc' 'unacc' 'unacc' 'acc' 'unacc'
 'unacc' 'acc' 'unacc' 'unacc' 'unacc' 'acc' 'unacc' 'acc' 'acc' 'acc'
 'acc' 'acc' 'acc' 'acc' 'unacc' 'acc' 'unacc' 'unacc' 'acc' 'acc' 'unacc'
 'acc' 'unacc' 'acc' 'acc' 'acc' 'acc' 'unacc' 'acc' 'acc' 'unacc' 'acc'
 'acc' 'acc' 'unacc' 'unacc' 'unacc' 'unacc' 'unacc' 'unacc' 'unacc'
 'unacc' 'unacc' 'unacc' 'acc' 'unacc' 'unacc' 'unacc' 'unacc' 'unacc'
 'acc' 'acc' 'acc' 'unacc' 'unacc' 'unacc' 'unacc' 'acc' 'unacc' 'unacc'
 'acc' 'acc' 'acc' 'unacc' 'acc' 'unacc' 'acc' 'unacc' 'unacc' 'unacc'
 'acc' 'unacc' 'unacc' 'acc' 'acc' 'unacc' 'unacc' 'unacc' 'unacc' 'acc'
 'unacc' 'unacc' 'unacc' 'unacc' 'acc' 'unacc' 'acc' 'acc' 'acc' 'unacc'
 'acc' 'unacc' 'unacc' 'acc' 'unacc' 'acc' 'unacc' 'unacc' 'unacc' 'unacc'
 'unacc' 'acc' 'unacc' 'acc' 'acc' 'acc' 'acc' 'acc' 'unacc

## TUGAS
Pada tugas kali ini Anda diminta memodifikasi metode pembentukan tree yang telah Anda agar metode tersebut menggunakan information gain sebagai dasar percabangan. Lengkapilah kerangka source code di bawah ini

Lengkapi fungsi hitung_entropy

In [ ]:
def hitung_entropy(kolom_kelas):
  elemen,banyak = np.unique(kolom_kelas,return_counts=True)
  prob = banyak/np.sum(banyak)
  entropy = -np.sum([p * np.log2(p) for p in prob if p > 0])
  return entropy

Lengkapi fungsi information_gain

In [ ]:
def information_gain(data, nama_fitur_split, nama_fitur_kelas):
  entropy_parent = hitung_entropy(data[nama_fitur_kelas])

  nilai_unik = np.unique(data[nama_fitur_split])
  total_data = len(data)

  entropy_split = 0

  for nilai in nilai_unik:
    subset = data[data[nama_fitur_split] == nilai]
    prob = len(subset) / total_data
    entropy_subset = hitung_entropy(subset[nama_fitur_kelas])
    entropy_split += prob * entropy_subset

  information_gain = entropy_parent - entropy_split

  return information_gain

Lengkapi fungsi **buat_tree_ig**. Isinya sama persis dengan fungsi **buat_tree**, hanya saja penghitungan **gini_split** diganti dengan **information_gain**. Selain itu, percabangan dilakukan dengan menggunakan nilai **information_gain** **terbesar**

In [ ]:
def buat_tree_ig(data,data_awal, daftar_fitur, nama_fitur_kelas,kelas_parent_node=None):
  #jika hanya ada satu kelas pada data
  if len(np.unique(data[nama_fitur_kelas])) <= 1:
    return np.unique(data[nama_fitur_kelas])[0]
  #jika data kosong
  elif len(data) == 0:
    return np.unique(data_awal[nama_fitur_kelas])[
      np.argmax(np.unique(data_awal[nama_fitur_kelas],return_counts=True)[1])]

  #jika tidak ada fitur yang terisa
  elif len(daftar_fitur) == 0:
    return kelas_parent_node
  else:
    kelas_parent_node = np.unique(data[nama_fitur_kelas])[
      np.argmax(np.unique(data[nama_fitur_kelas],return_counts=True)[1])]
    nilai_split = [information_gain(data,fitur,nama_fitur_kelas) for fitur in daftar_fitur]
    index_fitur_terbaik = np.argmax(nilai_split)
    fitur_terbaik = daftar_fitur[index_fitur_terbaik]
    tree = {fitur_terbaik:{}}
    daftar_fitur = [i for i in daftar_fitur if i != fitur_terbaik]
    for nilai in np.unique(data[fitur_terbaik]):
      sub_data = data.where(data[fitur_terbaik] == nilai).dropna()
      subtree = buat_tree(sub_data,data_awal,daftar_fitur,nama_fitur_kelas,
                          kelas_parent_node)
      tree[fitur_terbaik][nilai]=subtree
  return(tree)

Lakukan pembentukan tree menggunakan fungsi **buat_tree_ig**

In [ ]:
tree_ig = buat_tree_ig(data_latih,data_latih,data_latih.columns[:-1],'class')

Tampilkan tree yang terbentuk

In [ ]:
pprint(tree_ig)

{'safety': {'high': {'persons': {'2': 'unacc',
                                 '4': {'buying': {'high': {'maint': {'high': 'acc',
                                                                     'low': 'acc',
                                                                     'med': 'acc',
                                                                     'vhigh': 'unacc'}},
                                                  'low': {'maint': {'high': {'lug_boot': {'big': 'vgood',
                                                                                          'med': {'doors': {'2': 'acc',
                                                                                                            '3': 'acc',
                                                                                                            '4': 'vgood'}},
                                                                                          'small': 'acc'}},
                                    

Lakukan pengujian menggunakan tree yang terbentuk

In [ ]:
hasil_prediksi_total_ig = []
for i in range(len(data_uji_dict)):
  hasil_prediksi = prediksi(data_uji_dict[i],tree_ig)
  hasil_prediksi_total_ig.append(hasil_prediksi)
print("Total prediksi benar: ",sum(hasil_prediksi_total_ig==data_uji['class']))
print("Total prediksi salah: ",sum(hasil_prediksi_total_ig!=data_uji['class']))

Total prediksi benar:  298
Total prediksi salah:  48


### PERTANYAAN

Jawablah pertanyaan di bawah ini



1.   Amati tree yang dihasilkan dengan kriteria percabangan GINI dan Information Gain. Apa perbedaan tree yang dihasilkan dari kedua metode tersebut?
2.   Apakah penggunaan Information Gain dapat meningkatkan akurasi prediksi?



Tulis jawaban Anda di cell ini


1.   Pada percobaan ini, tree yang dihasilkan oleh metode Gini dan Information Gain ternyata sama. Hal ini terjadi karena dataset memiliki pola yang jelas sehingga kedua metode memilih fitur terbaik yang sama pada setiap percabangan. Akibatnya, struktur tree, urutan percabangan, dan hasil klasifikasi menjadi identik.

2.   Penggunaan Information Gain tidak selalu meningkatkan akurasi prediksi. Pada kasus ini, karena struktur tree yang dihasilkan sama dengan metode Gini, maka akurasi yang dihasilkan juga cenderung sama. Hal ini menunjukkan bahwa pemilihan metode split tidak selalu mempengaruhi performa jika data memiliki pola yang kuat dan jelas.

